# Error Taxonomy Analysis - Real-World Episodes

This notebook mirrors the simulated analysis workflow for real-world runs using episode state summaries.

Two-round workflow:
1. Round 1 - Triage: review each episode with key evidence and assign a taxonomy code quickly.
2. Round 2 - Deep dive: re-open episodes marked NEEDS-REVIEW with richer context (L1, L2, and llm trace snippets).

### Error Mode Taxonomy (Root-Cause)

| Code | Layer | Root cause |
|------|-------|------------|
| PER-MISSING | Perception | Relevant state never captured in observations |
| PER-SPATIAL | Perception | Wrong spatial relation assignment (inside/on/near/etc.) |
| CTX-HIDDEN | Context | Signal appears in L1 but is weak/absent in L2 context |
| CTX-AFFORD | Context | L2 context present but too generic for grounding/preconditions |
| RSN-DIAG | Reasoning | Failure diagnosis is incorrect despite available evidence |
| RSN-PLAN | Reasoning | Diagnosis reasonable but corrective plan is flawed/incomplete |
| EXEC-REAL | Execution | Real-world execution/control slip despite valid reasoning |
| NEEDS-REVIEW | Admin | Insufficient evidence in triage; requires deep inspection |
| OTHER | Admin | Outlier not covered by the model |


In [ ]:
import ast
import json
import os
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import HTML, clear_output, display


In [ ]:
# Paths (run notebook from reflect/real_world_code)
STATE_SUMMARY_DIR = Path('./real_world/state_summary')
FINAL_RESULTS_CSV = Path('./real_world/final_results_with_explanations.csv')
HUMAN_EVAL_CSV = Path('./real_world/failure_explanation_evaluation.csv')
ROUND1_OUTPUT = Path('./real_world/error_taxonomy_real_round1.csv')
ROUND2_OUTPUT = Path('./real_world/error_taxonomy_real_round2.csv')

TAXONOMY_CODES = [
    'PER-MISSING',
    'PER-SPATIAL',
    'CTX-HIDDEN',
    'CTX-AFFORD',
    'RSN-DIAG',
    'RSN-PLAN',
    'EXEC-REAL',
    'NEEDS-REVIEW',
    'OTHER',
]

TAXONOMY_DESCRIPTIONS = {
    'PER-MISSING': 'Relevant state was never captured in observations.',
    'PER-SPATIAL': 'Objects were detected but spatial relationships were wrong.',
    'CTX-HIDDEN': 'Evidence appears in richer context but is missing/weakened in decision context.',
    'CTX-AFFORD': 'State is visible but still underspecified to ground required affordance/precondition.',
    'RSN-DIAG': 'Model had evidence but misdiagnosed failure cause or step.',
    'RSN-PLAN': 'Diagnosis is reasonable but correction strategy is logically flawed.',
    'EXEC-REAL': 'Likely motor/control/embodiment failure in real execution.',
    'NEEDS-REVIEW': 'Not enough evidence in triage; inspect in detail.',
    'OTHER': 'Outlier that does not fit current taxonomy.',
}


In [ ]:
def _safe_read_json(path: Path):
    try:
        with path.open('r') as f:
            return json.load(f)
    except Exception:
        return {}


def _safe_literal_eval(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    s = str(x).strip()
    if not s:
        return []
    try:
        obj = ast.literal_eval(s)
        return obj if isinstance(obj, list) else [obj]
    except Exception:
        return [s]


def _first_time_token(text: str):
    if not isinstance(text, str):
        return None
    m = re.search(r'\b(\d{1,2}:\d{2})\b', text)
    return m.group(1) if m else None


def _to_seconds(ts):
    if not isinstance(ts, str) or ':' not in ts:
        return None
    p = ts.split(':')
    if len(p) != 2:
        return None
    try:
        return int(p[0]) * 60 + int(p[1])
    except ValueError:
        return None


def _loc_hit(pred_steps, gt_steps):
    pred_secs = [_to_seconds(s) for s in pred_steps]
    gt_secs = [_to_seconds(s) for s in gt_steps]
    pred_secs = [s for s in pred_secs if s is not None]
    gt_secs = [s for s in gt_secs if s is not None]
    if not pred_secs or not gt_secs:
        return np.nan
    if len(gt_secs) == 1:
        return any(p == gt_secs[0] for p in pred_secs)
    lo, hi = min(gt_secs), max(gt_secs)
    return any(lo <= p <= hi for p in pred_secs)


def _contains_any(text, keywords):
    text = str(text).lower()
    return any(k in text for k in keywords)


In [ ]:
# Build episode table from state_summary/*/reasoning.json
rows = []
for ep_dir in sorted(STATE_SUMMARY_DIR.iterdir()):
    if not ep_dir.is_dir():
        continue
    if ep_dir.name == 'batch_timings':
        continue

    reasoning_path = ep_dir / 'reasoning.json'
    llm_trace_path = ep_dir / 'llm_trace.json'
    r = _safe_read_json(reasoning_path)
    trace = _safe_read_json(llm_trace_path)

    pred_steps = _safe_literal_eval(r.get('pred_failure_step', []))
    gt_steps = _safe_literal_eval(r.get('gt_failure_step', []))

    row = {
        'episode': ep_dir.name,
        'task': re.sub(r'\d+$', '', ep_dir.name),
        'reasoning_file': str(reasoning_path),
        'llm_trace_file': str(llm_trace_path),
        'pred_failure_reason': r.get('pred_failure_reason', ''),
        'pred_failure_step': pred_steps,
        'gt_failure_reason': r.get('gt_failure_reason', ''),
        'gt_failure_step': gt_steps,
        'l1_summary': r.get('l1_summary', ''),
        'l2_summary': r.get('l2_summary', ''),
        'llm_trace_count': len(trace) if isinstance(trace, dict) else 0,
    }
    row['Loc'] = _loc_hit(pred_steps, gt_steps)
    rows.append(row)

df = pd.DataFrame(rows).sort_values(['task', 'episode']).reset_index(drop=True)
print(f'Episodes loaded from state_summary: {len(df)}')
df.head(3)


In [ ]:
# Merge with available evaluation/result files
if FINAL_RESULTS_CSV.exists():
    fr = pd.read_csv(FINAL_RESULTS_CSV)
    fr = fr.rename(columns={'folder': 'episode'})
    keep_cols = [c for c in ['episode', 'task_id', 'loc_hit', 'human_score'] if c in fr.columns]
    if keep_cols:
        df = df.merge(fr[keep_cols].drop_duplicates('episode'), on='episode', how='left', suffixes=('', '_fr'))

if HUMAN_EVAL_CSV.exists():
    he = pd.read_csv(HUMAN_EVAL_CSV)
    he = he.rename(columns={'task': 'episode'})
    keep_cols = [c for c in ['episode', 'human_score'] if c in he.columns]
    if keep_cols:
        he = he[keep_cols].drop_duplicates('episode')
        he = he.rename(columns={'human_score': 'human_score_eval'})
        df = df.merge(he, on='episode', how='left')

if 'loc_hit' in df.columns:
    df['Loc'] = df['Loc'].fillna(df['loc_hit'])

if 'human_score_eval' in df.columns:
    df['human_score'] = df.get('human_score', np.nan)
    df['human_score'] = df['human_score'].fillna(df['human_score_eval'])

df['Exp'] = df['human_score'].apply(lambda x: np.nan if pd.isna(x) else bool(int(x)))

print('Loc known:', int(df['Loc'].notna().sum()), '/', len(df))
print('Exp known:', int(df['Exp'].notna().sum()), '/', len(df))
display(df[['episode', 'task', 'Loc', 'Exp', 'pred_failure_step', 'gt_failure_step']].head(10))


In [ ]:
# Simple automatic suggestion heuristic (editable by human)
def suggest_taxonomy(row):
    gt = str(row.get('gt_failure_reason', '')).lower()
    pred = str(row.get('pred_failure_reason', '')).lower()
    l1 = str(row.get('l1_summary', '')).lower()
    l2 = str(row.get('l2_summary', '')).lower()
    loc = row.get('Loc', np.nan)
    exp = row.get('Exp', np.nan)

    impossible_markers = [
        'inside', 'on top of', 'simultaneously', 'impossible', 'overlapping'
    ]
    drop_markers = ['drop', 'dropped', 'slip', 'could not grasp', 'gripper remained empty']

    if _contains_any(gt + ' ' + pred, ['dropped', 'drop', 'slip', 'accidentally']) and _contains_any(l2, drop_markers):
        return 'EXEC-REAL', 'Drop/slip pattern suggests real-world execution/control failure.'

    if _contains_any(l2, impossible_markers) and _contains_any(pred, ['inside', 'overlapping', 'impossible', 'nested']):
        return 'PER-SPATIAL', 'Contradictory spatial relations indicate relation-assignment perception error.'

    if (not pd.isna(loc) and bool(loc)) and (not pd.isna(exp) and bool(exp)):
        return 'RSN-PLAN', 'Failure step/explanation alignment is strong; likely planning or correction flaw.'

    if (not pd.isna(loc) and not bool(loc)) or (not pd.isna(exp) and not bool(exp)):
        return 'RSN-DIAG', 'Mismatch on location/explanation indicates diagnosis error.'

    if len(l1) > len(l2) * 1.2 and not _contains_any(l2, ['auditory observation']) and _contains_any(l1, ['action:', 'visual observation']):
        return 'CTX-HIDDEN', 'Potential context compression from L1 to L2 may hide critical evidence.'

    if _contains_any(gt, ['should', 'instead of', 'before']) and _contains_any(pred, ['failed to', 'could not']):
        return 'CTX-AFFORD', 'Precondition/affordance grounding likely insufficient.'

    if _contains_any(pred, ['nothing is inside robot gripper']) and _contains_any(gt, ['blocking', 'occupied', 'already inside']):
        return 'PER-MISSING', 'Relevant actionable object state likely not captured for grasp/planning.'

    return 'NEEDS-REVIEW', 'No confident rule matched; needs manual inspection.'


sugg = df.apply(lambda r: suggest_taxonomy(r), axis=1)
df['taxonomy_suggested'] = [x[0] for x in sugg]
df['suggestion_reason'] = [x[1] for x in sugg]

display(df[['episode', 'taxonomy_suggested', 'suggestion_reason']].head(12))


In [ ]:
# Round 1 triage widget
class TriageUI:
    def __init__(self, frame, output_csv):
        self.df = frame.copy().reset_index(drop=True)
        self.output_csv = Path(output_csv)

        self.i = 0
        self.dd = widgets.Dropdown(options=TAXONOMY_CODES, description='Code:')
        self.note = widgets.Textarea(description='Note:', layout=widgets.Layout(width='100%', height='90px'))
        self.save_btn = widgets.Button(description='Save Label', button_style='success')
        self.prev_btn = widgets.Button(description='Prev')
        self.next_btn = widgets.Button(description='Next')
        self.jump = widgets.BoundedIntText(description='Go to', min=1, max=max(1, len(self.df)), value=1)
        self.jump_btn = widgets.Button(description='Jump')
        self.out = widgets.Output()

        self.save_btn.on_click(self._save)
        self.prev_btn.on_click(self._prev)
        self.next_btn.on_click(self._next)
        self.jump_btn.on_click(self._jump)

        if 'taxonomy_final' not in self.df.columns:
            self.df['taxonomy_final'] = self.df['taxonomy_suggested']
        if 'taxonomy_note' not in self.df.columns:
            self.df['taxonomy_note'] = ''

    def _render(self):
        clear_output(wait=True)
        row = self.df.iloc[self.i]
        self.dd.value = row['taxonomy_final'] if row['taxonomy_final'] in TAXONOMY_CODES else row['taxonomy_suggested']
        self.note.value = str(row.get('taxonomy_note', ''))

        status = f"Episode {self.i + 1}/{len(self.df)} | {row['episode']}"
        evidence = f"""
<h4>{status}</h4>
<b>Task:</b> {row['task']}<br>
<b>Loc:</b> {row['Loc']} | <b>Exp:</b> {row['Exp']}<br>
<b>Suggested:</b> {row['taxonomy_suggested']}<br>
<b>Suggestion reason:</b> {row['suggestion_reason']}<br><br>
<b>GT failure reason</b><br>{row['gt_failure_reason']}<br><br>
<b>Pred failure reason</b><br>{row['pred_failure_reason']}<br><br>
<details><summary><b>L2 summary</b></summary><pre>{row['l2_summary']}</pre></details>
<details><summary><b>L1 summary</b></summary><pre>{row['l1_summary']}</pre></details>
"""
        display(HTML(evidence))
        display(widgets.HBox([self.prev_btn, self.next_btn, self.jump, self.jump_btn]))
        display(self.dd)
        display(self.note)
        display(self.save_btn)
        display(self.out)

    def _save(self, _):
        self.df.loc[self.i, 'taxonomy_final'] = self.dd.value
        self.df.loc[self.i, 'taxonomy_note'] = self.note.value
        self.df.to_csv(self.output_csv, index=False)
        with self.out:
            clear_output(wait=True)
            print(f'Saved: {self.df.iloc[self.i]["episode"]} -> {self.dd.value}')

    def _prev(self, _):
        self.i = max(0, self.i - 1)
        self._render()

    def _next(self, _):
        self.i = min(len(self.df) - 1, self.i + 1)
        self._render()

    def _jump(self, _):
        self.i = int(np.clip(self.jump.value - 1, 0, len(self.df) - 1))
        self._render()

    def show(self):
        self._render()


triage = TriageUI(df, ROUND1_OUTPUT)
triage.show()


In [ ]:
# Round 2 deep dive for NEEDS-REVIEW
if ROUND1_OUTPUT.exists():
    round1 = pd.read_csv(ROUND1_OUTPUT)
else:
    round1 = df.copy()

pending = round1[round1['taxonomy_final'] == 'NEEDS-REVIEW'].copy().reset_index(drop=True)
print(f'Episodes marked NEEDS-REVIEW: {len(pending)}')

if len(pending) == 0:
    print('No deep-dive episodes right now.')
else:
    pending['llm_trace_preview'] = ''
    for i, r in pending.iterrows():
        trace = _safe_read_json(Path(r['llm_trace_file']))
        if isinstance(trace, dict) and trace:
            chunks = []
            for k in sorted(trace.keys())[:3]:
                obj = trace[k]
                prompt_user = obj.get('prompt', {}).get('user', '') if isinstance(obj, dict) else ''
                response = obj.get('response', '') if isinstance(obj, dict) else ''
                chunks.append(f'[{k}]\nUSER: {prompt_user}\nRESPONSE: {response}')
            pending.loc[i, 'llm_trace_preview'] = '\n\n'.join(chunks)

    deep_dive_cols = [
        'episode', 'task', 'Loc', 'Exp', 'taxonomy_suggested', 'taxonomy_final',
        'gt_failure_reason', 'pred_failure_reason', 'l2_summary', 'l1_summary',
        'llm_trace_preview', 'taxonomy_note'
    ]
    display(pending[deep_dive_cols])

    pending.to_csv(ROUND2_OUTPUT, index=False)
    print(f'Round 2 table exported to: {ROUND2_OUTPUT}')


In [ ]:
# Summary charts from round1 labels
src = pd.read_csv(ROUND1_OUTPUT) if ROUND1_OUTPUT.exists() else df
label_col = 'taxonomy_final' if 'taxonomy_final' in src.columns else 'taxonomy_suggested'

cnt = src[label_col].value_counts().reindex(TAXONOMY_CODES, fill_value=0)
cnt = cnt.drop(['NEEDS-REVIEW', 'OTHER', 'EXEC-REAL', 'CTX-HIDDEN', 'CTX-AFFORD', 'RSN-PLAN'], errors='ignore')

display(cnt.to_frame('count'))

plt.figure(figsize=(10, 4))
cnt.plot(kind='bar')
for i, v in enumerate(cnt):
    plt.text(i, v + 0.5, str(v), ha='center', va='bottom')
plt.title('Real-World Error Taxonomy Counts')
plt.ylabel('Episodes')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

if 'task' in src.columns:
    pivot = pd.crosstab(src['task'], src[label_col])
    display(pivot)


In [ ]:
# Create Pie Chart for Taxonomy Distribution grouped by category
pie_data = cnt.drop(
    ['NEEDS-REVIEW', 'OTHER', 'EXEC-REAL', 'CTX-HIDDEN', 'CTX-AFFORD', 'RSN-PLAN'],
    errors='ignore'
)

category_map = {
    'PER-MISSING': 'Perception',
    'PER-SPATIAL': 'Perception',
    'RSN-DIAG': 'Reasoning'
}

# 1) map taxonomy -> category
# 2) sum duplicated category names (e.g., Perception)
pie_data = pie_data.rename(index=category_map).groupby(level=0).sum()

plt.figure(figsize=(6, 6))
plt.pie(
    pie_data.values,
    labels=pie_data.index,
    autopct='%1.1f%%',
    startangle=140,
    colors=['#ff9999', '#66b3ff', '#99ff99', '#ffcc99'][:len(pie_data)]
)
plt.title('Distribution of Failure Taxonomy Categories')
plt.axis('equal')
plt.tight_layout()
plt.show()